# Impala67 AI – QLoRA Finetuning (Google Colab)
Trainiert **Qwen 3.5 4B** (oder 2B) mit Unsloth. Läuft auf T4 (fp16) und A100/L4 (bf16).

**Dateien kommen automatisch aus Google Drive**: Ordner `Meine Ablage/Impala67-Training/` mit
`chat_format.py`, `generate_dataset.py`, `tool_defs.json`. Zelle 0 kopiert sie nach `/content`
und erzeugt den Datensatz dort neu (fester Zufalls-Seed → identisch zum lokal geprüften Datensatz).

## 0. Dateien aus Google Drive holen und Datensatz erzeugen
Fragt beim ersten Mal nach der Drive-Freigabe.

In [ ]:
import os, shutil, subprocess
from google.colab import drive

drive.mount("/content/drive")
SRC = "/content/drive/MyDrive/Impala67-Training"
NEEDED = ["chat_format.py", "generate_dataset.py", "tool_defs.json"]

os.chdir("/content")
missing = [f for f in NEEDED if not os.path.exists(f"{SRC}/{f}")]
assert not missing, f"Fehlt in {SRC}: {missing}"
for f in NEEDED:
    shutil.copy(f"{SRC}/{f}", f)

# Datensatz hier erzeugen (landet in /content/data/) und nach /content legen
print(subprocess.run(["python", "generate_dataset.py"], capture_output=True, text=True, check=True).stdout)
for f in ["impala_tools_train.jsonl", "impala_tools_val.jsonl"]:
    shutil.copy(f"data/{f}", f)
print("Alle Dateien da.")

## 1. Unsloth installieren

In [ ]:
%%capture
!pip install unsloth

## 2. Modell laden (4-Bit) und LoRA-Adapter anlegen

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
model_name = "unsloth/Qwen3.5-4B"  # Alternativ: "unsloth/Qwen3.5-2B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Das Chat-Template des Tokenizers bleibt unverändert (Qwen-eigenes Tool- und Thinking-Format).
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 3. Daten mit dem Qwen-Template rendern

In [ ]:
from datasets import Dataset
from chat_format import INSTRUCTION_PART, RESPONSE_PART, build_texts, load_tools

tools = load_tools("tool_defs.json")
train_ds = Dataset.from_dict({"text": build_texts(tokenizer, "impala_tools_train.jsonl", tools, max_seq_length)})
val_ds = Dataset.from_dict({"text": build_texts(tokenizer, "impala_tools_val.jsonl", tools, max_seq_length)})
print(f"Training: {len(train_ds)} | Validierung: {len(val_ds)}")

# Sichtprüfung: Tools im Systemteil, <tool_call><function=...>, ein <think>-Block je Antwort
print(train_ds[0]["text"][-1500:])

## 4. Training (3 Epochen, Loss nur auf Assistenten-Antworten)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

bf16 = torch.cuda.is_bf16_supported()  # T4: False -> fp16

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        warmup_ratio = 0.05,
        learning_rate = 1e-4,
        bf16 = bf16,
        fp16 = not bf16,
        logging_steps = 5,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        save_strategy = "no",
        report_to = "none",
    ),
)

# Systemprompt, Tools, Skripte und Tool-Ergebnisse zählen nicht zum Loss.
trainer = train_on_responses_only(trainer, instruction_part = INSTRUCTION_PART, response_part = RESPONSE_PART)

trainer_stats = trainer.train()

## 5. Kurztest vor dem Export
Muss einen `<tool_call>` mit `<function=change>` und `page.patch` erzeugen – nicht sofort stoppen.

In [ ]:
FastLanguageModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
msgs = [
    {"role": "system", "content": "Du bist der KI-Coach von Impala67, einer lokalen Notiz- und Lern-App. Antworte auf Deutsch, kompakt."},
    {"role": "user", "content": "Korrigiere in 'Physik 1' bitte 'v = s * t' zu '$v = s / t$'."},
]
prompt = tok.apply_chat_template(msgs, tools = tools, tokenize = False, add_generation_prompt = True)
inputs = tok(prompt, return_tensors = "pt").to("cuda")
out = model.generate(**inputs, max_new_tokens = 300, do_sample = False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:]))

## 6. GGUF exportieren (Q4_K_M) und herunterladen
Das Qwen-Template wandert unverändert in die GGUF-Metadaten. Hinweis: Es wird nur das Sprachmodell exportiert, kein `mmproj` – Bilder gehen damit in LM Studio nicht (Benchmark mit `--no-vision`).

In [ ]:
model.save_pretrained_gguf("impala67_model", tokenizer, quantization_method = "q4_k_m")

from google.colab import files
import glob

for f in glob.glob("impala67_model*/*.gguf") + glob.glob("*.gguf"):
    print(f"Lade {f} herunter ...")
    files.download(f)